In [ ]:
# Developed and tested on Python 3.13
#
# Before running the notebook, create a collection in MongoDB and import both files in the chunked_docs folder,
# using either the command line tool mongoimport or the "Add data / import JSON file" feature in MongoDB Compass.
# In my case, I'll call the database "test_voyage_rag" and the collection "mongodb_search_kb"
#
# Also, before running this notebook, install the dependencies with the command:
# pip install pymongo==4.16.0 voyageai==0.3.7

In [ ]:
from pymongo import MongoClient
import pymongo
from pymongo.operations import SearchIndexModel
from bson.binary import Binary, BinaryVectorDtype

import voyageai

from getpass import getpass

## Configurations

In [ ]:
mongo_conn_string = getpass("Type or paste your MongoDB connection string")
client = MongoClient(mongo_conn_string)

In [ ]:
mongo_db = client['test_voyage_rag']
kb_coll = mongo_db['mongodb_search_kb']

In [ ]:
voyage_api_key = getpass("Type or paste your Voyage AI API Key")
voyage_client = voyageai.Client(voyage_api_key)

In [ ]:
def get_embeddings(input_docs, input_type):
    response = voyage_client.embed(
        input_docs,
        model="voyage-4",
        input_type=input_type,
        output_dtype="float",
        output_dimension = 1024
    )

    return response.embeddings

## Step 1: compute embeddings and load documents to DB

In [ ]:
def process_docs_batch(temp_batch):
    # Compute embeddings for all documents in the batch
    fullplot_embeddings = get_embeddings(
        input_docs = [ x['content'] for x in temp_batch], 
        input_type = "document"
    )

    # Create an upsert operation for each document - all upserts are later run as a bulk operation. 
    mongo_ops_list = []
    for i in range(len(temp_batch)):
        temp_doc = temp_batch[i]

        # Use the composed key (file_relative_path, chunk_no) as replace key when upserting documents to the database.
        upsert_filter = {
            "file_relative_path": temp_doc['file_relative_path'],
            "chunk_no": temp_doc['chunk_no']
        }


        # Add the embedding vector to the original document, forcing the use of FLOAT32 binary type.
        temp_binary_embedding = Binary.from_vector(
            fullplot_embeddings[i], 
            BinaryVectorDtype(BinaryVectorDtype.FLOAT32)
        )
        temp_doc['voyage4-float'] = temp_binary_embedding
        
        mongo_ops_list.append(
            pymongo.ReplaceOne(
                filter=upsert_filter,  
                replacement=temp_doc,
                upsert=True
            )
        )
    
    # Run all operations in a single unordered bulk to optimize latency
    kb_coll.bulk_write(mongo_ops_list, ordered=False)


# Create an index on the composed key (file_relative_path, chunk_no), to optimize latency
kb_coll.create_index(
    [
        ("file_relative_path", pymongo.ASCENDING), 
        ("chunk_no", pymongo.ASCENDING), 
    ], 
    unique=True
)

# Retrieve all documents with missing embeddings
results = kb_coll.find(
    filter = {
        "voyage4-float":{"$exists": False}
    },
    projection = {"_id":False}
)

temp_batch = []
for temp_movie in results:
    # Process documents in batches of 100
    temp_batch.append(temp_movie)
    if len(temp_batch) < 100:
        continue

    process_docs_batch(temp_batch)
    temp_batch = []

if len(temp_batch)>0:
    process_docs_batch(temp_batch)

## Step 2: create the MongoDB Search indexes

In [ ]:
# Vector Index configuration
vector_index_model = SearchIndexModel(
    name="test_voyage_rag_vector",
    type="vectorSearch",
    definition={
        "fields": [
            {
                "type": "vector",
                "path": "voyage4-float",
                "numDimensions": 1024,

                "indexingMethod": "hnsw",
                "similarity": "dotProduct",

                "quantization": "binary",
            }
        ]
    }
)

# Apply the index definition above
kb_coll.create_search_index(model=vector_index_model)

In [ ]:
fulltext_search_model = SearchIndexModel(
    name="test_voyage_rag_fulltext",
    type="search",
    definition={
        "mappings": {
            "dynamic": False,
            "fields": {
                "content": {
                    "type": "string",
                    "indexOptions": "docs",
                    "norms": "omit"
                }
            }
        }
    }
)

# Apply the index definition above
kb_coll.create_search_index(model=fulltext_search_model)

## Step 3: run a Hybrid Search query
Let's test our setup by running a hybrid search query, that uses both the vector index as well as the full-text search index.

> NOTE: Before running the query, wait a minute after running the steps above to make sure the creation of the indexes is completed.

> NOTE 2: MongoDB 8.3 or greater is required to run hybrid search with $scoreFusion as per pipeline below.

You may use MongoDB Compass to check the status of the indexes: wait until they become "READY".

In [ ]:
plaintext_query = """
Quale provider viene usato in MongoDB Search per la funzione di autoEmbed, ovvero per abilitare la generazione automatizzata di text embedding sui documenti nel database?

Opzioni:
- A) OpenAI
- B) Voyage AI
- C) Hugging Face
- D) AWS Bedrock
"""

# Get the embedding of the query
query_embedding = get_embeddings(
    input_docs = [plaintext_query], 
    input_type = "query"
)[0]

result = kb_coll.aggregate([
  {
    "$scoreFusion": {
      "input": {
        "pipelines": {
          "vectorPipeline": [
            {
              "$vectorSearch": {
              "index": "test_voyage_rag_vector",
              "path": "voyage4-float",
              "queryVector": query_embedding,
              "numCandidates": 100,
              "limit": 20
              }
            }
          ],
          "fullTextPipeline": [
            {
              "$search": {
                "index": "test_voyage_rag_fulltext",
                "text": {
                  "query": plaintext_query,
                  "path": "content"
                }
              }
            },
            { "$limit": 20 }
          ]
        },
        "normalization": "minMaxScaler"
      },
      "combination": {
        "weights": {
          "vectorPipeline": 1,
          "fullTextPipeline": 1
        }
      },
      "scoreDetails": True
    }
  },
  {
    "$project": {
      "_id": 0,
      "content": 1,
      "scoreDetails": {"$meta": "score"}
    }
  },
  {
    "$limit": 10
  }
]
)

test_results = [x for x in result]
test_results